# SQL Query Optimization — AI Engineering Interview Prep

Covers: EXPLAIN, indexes, anti-patterns, query rewriting in DuckDB.

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import time

conn = duckdb.connect()

# Large dataset for meaningful benchmarks
np.random.seed(42)
n = 500_000
df = pd.DataFrame({
    'id': range(n),
    'user_id': np.random.randint(1, 10000, n),
    'amount': np.random.uniform(1, 1000, n).round(2),
    'category': np.random.choice(['A','B','C','D','E'], n),
    'created_at': pd.date_range('2022-01-01', periods=n, freq='1min')
})
conn.execute("CREATE TABLE transactions AS SELECT * FROM df")
print(f"Created transactions table with {n:,} rows")

## 1. EXPLAIN — Execution Plan

In [ ]:
# DuckDB EXPLAIN shows physical plan
plan = conn.execute("""
EXPLAIN
SELECT category, COUNT(*), SUM(amount)
FROM transactions
WHERE amount > 500
GROUP BY category
""").fetchall()

for line in plan:
    print(line[1])

## 2. Timing Queries

In [ ]:
def timed_query(label, sql):
    start = time.perf_counter()
    result = conn.execute(sql).df()
    elapsed = time.perf_counter() - start
    print(f"{label}: {elapsed*1000:.1f}ms  ({len(result)} rows)")
    return result

# Simple aggregation
timed_query("Category aggregate",
    "SELECT category, COUNT(*), ROUND(AVG(amount),2) FROM transactions GROUP BY category")

# Filter then aggregate
timed_query("Filter + aggregate",
    "SELECT category, COUNT(*) FROM transactions WHERE amount > 750 GROUP BY category")

## 3. Anti-Patterns

In [ ]:
# Anti-pattern 1: SELECT * — retrieves unneeded columns
t1 = time.perf_counter()
conn.execute("SELECT * FROM transactions WHERE category = 'A'").df()
t_star = time.perf_counter() - t1

t2 = time.perf_counter()
conn.execute("SELECT id, amount FROM transactions WHERE category = 'A'").df()
t_cols = time.perf_counter() - t2

print(f"SELECT *:          {t_star*1000:.1f}ms")
print(f"SELECT id,amount:  {t_cols*1000:.1f}ms")
print(f"DuckDB is columnar so column selection can be significant in wide tables.")

In [ ]:
# Anti-pattern 2: Functions on columns prevent predicate pushdown
# Bad: WHERE UPPER(category) = 'A'  — must evaluate UPPER for every row
# Good: WHERE category = 'a' (after normalizing data at insert time)

t1 = time.perf_counter()
r1 = conn.execute("SELECT COUNT(*) FROM transactions WHERE UPPER(category) = 'A'").fetchone()[0]
bad_time = time.perf_counter() - t1

t2 = time.perf_counter()
r2 = conn.execute("SELECT COUNT(*) FROM transactions WHERE category = 'A'").fetchone()[0]
good_time = time.perf_counter() - t2

print(f"UPPER() on column: {bad_time*1000:.1f}ms  → {r1:,} rows")
print(f"Direct compare:    {good_time*1000:.1f}ms  → {r2:,} rows")
print("Rule: avoid applying functions to the column being filtered.")

## 4. Query Rewriting — Subquery vs CTE vs JOIN

In [ ]:
# Find transactions with amount above average for their category

# Version 1: Correlated subquery (runs once per row)
timed_query("Correlated subquery", """
SELECT id, category, amount
FROM transactions t
WHERE amount > (SELECT AVG(amount) FROM transactions t2 WHERE t2.category = t.category)
LIMIT 100
""")

# Version 2: CTE (materialise averages once)
timed_query("CTE approach", """
WITH cat_avg AS (
    SELECT category, AVG(amount) AS avg_amt
    FROM transactions GROUP BY category
)
SELECT t.id, t.category, t.amount
FROM transactions t
JOIN cat_avg c ON t.category = c.category
WHERE t.amount > c.avg_amt
LIMIT 100
""")

## 5. EXPLAIN ANALYZE

In [ ]:
plan = conn.execute("""
EXPLAIN ANALYZE
SELECT user_id, COUNT(*), SUM(amount)
FROM transactions
WHERE category IN ('A', 'B') AND amount > 100
GROUP BY user_id
HAVING COUNT(*) > 5
""").fetchall()

for line in plan:
    print(line[1])

## Interview Tips

- **Columnar databases** (DuckDB, BigQuery, Redshift) benefit hugely from selecting fewer columns.
- **Filter early**: push WHERE conditions as close to the data source as possible.
- **CTE materialisation**: in some DBs (PostgreSQL), CTEs act as optimisation fences. Use subqueries in hot paths.
- **Index on low-cardinality columns** (like category with 5 values) is often worse than a full scan.
- **EXPLAIN vs EXPLAIN ANALYZE**: EXPLAIN shows the plan; ANALYZE actually runs it and shows true costs.
- **Partitioning** by date column dramatically reduces scanned data for time-range queries in data warehouses.